# Lab 16: Vector and hybrid retrieval

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


One chunk per block here (the sample is small). Scope is enforced inside the store and on the lexical side.

In [2]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [3]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [4]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex)[:k]

In [5]:
hits = hybrid("customer concentration", embed("customer concentration"), scope="deal_a", k=5)
print(hits)
assert all(not h.startswith("DR-deal_b") for h in hits)
try:
    search(embed("customer concentration"), scope="")
except ValueError as e:
    print("as expected:", e)

['DR-deal_a-001:p0003_b001', 'DR-deal_a-002:t002', 'DR-deal_a-003:p0001_b003', 'DR-deal_a-001:p0004_b002', 'DR-deal_a-001:p0002_b001']
as expected: scope is required


**Exercise 16.** Compare `rrf_k` of 10 and 60 on the qrels in notebook 17. Does the ranking change?

In [6]:
import pandas as pd

qrels = [
    json.loads(line)
    for line in open("data/eval/qrels.jsonl")
]

comparisons = []

for q in qrels:
    # Retrieve candidates once so both RRF settings use the same inputs.
    dense = search(
        embed(q["query"]),
        q["scope"],
        k=min(50, len(chunks)),
    )["ids"][0]

    scores = bm25.get_scores(tok(q["query"]))
    lexical = [
        chunks[i]["chunk_id"]
        for i in scores.argsort()[::-1]
        if chunks[i]["deal_id"] in (q["scope"], "public")
    ][:50]

    ranking_10 = rrf(dense, lexical, k=10)
    ranking_60 = rrf(dense, lexical, k=60)

    comparisons.append({
        "query": q["query"],
        "ranking_changed": ranking_10 != ranking_60,
        "top5_changed": ranking_10[:5] != ranking_60[:5],
        "top5_rrf10": ranking_10[:5],
        "top5_rrf60": ranking_60[:5],
    })

comparison = pd.DataFrame(comparisons)
display(comparison[["query", "ranking_changed", "top5_changed"]])

print("Full rankings changed:", comparison["ranking_changed"].sum())
print("Top-five rankings changed:", comparison["top5_changed"].sum())

,query,ranking_changed,top5_changed
0,What was Wren's FY2025 revenue?,True,False
1,How fast did Wren grow in FY2025?,True,True
2,When does Wren's fiscal year end?,True,False
3,How concentrated is the customer base?,False,False
4,What is net revenue retention?,True,False
5,Which contracts have change-of-control provisi...,True,True
6,What are the adjusted EBITDA add-backs?,True,False
7,Who is Wren's CFO?,True,False
8,When are indications of interest due?,True,False
9,What revenue does the run-rate claim state?,False,False


Full rankings changed: 9
Top-five rankings changed: 2
